# Iteration 3 — feature extraction (module 3)

Starting point = RESULTS.md row 2.4: nights cropped to the sleep period ±30 min, EEG band-pass 0.3–35 Hz,
the 11 supplied features, `imbalance="none"`, random forest, 3 subjects, leave-one-subject-out, seed 0.

Each row 3.1–3.6 adds **one** feature group (`cfg["feature_groups"]`, code in `tracks/sleep_features.py`);
row 3.7 combines the groups that passed the rule written in RESULTS.md **before** running.
Comparison budget: 7 configurations — nothing else is tried on these folds.

In [1]:
import sys, os
for sub in ("tracks", "src", "src/bsp"):
    sys.path.insert(0, os.path.abspath(os.path.join("..", sub)))
import numpy as np
from sklearn.metrics import recall_score
from sleep_edf import SleepEDFTrack
from sleep_crop import crop_to_sleep

track = SleepEDFTrack()
cache = os.path.join("data_cache", "sleep_edf")       # run from notebooks/
recs = [crop_to_sleep(r) for r in track.load(cache)]  # row 2.2 cropping
print("nights:", len(recs), "| subjects:", sorted({r.group for r in recs}))
print("channels:", list(recs[0].epochs))

nights: 6 | subjects: ['SC400', 'SC401', 'SC402']
channels: ['eeg', 'eog', 'emg', 'eeg2']


## Signal check before choosing features

`EMG submental` in Sleep-Cassette is stored at 1 Hz (an amplitude envelope), so only level-type EMG
features make sense. Pz-Oz is stored at 100 Hz but the supplied loader never read it — `load()` now
keeps it as `epochs["eeg2"]` (only the `pz` group uses it).

In [2]:
import mne, glob
psg = sorted(glob.glob(os.path.join(cache, "**", "*PSG.edf"), recursive=True))[0]
raw = mne.io.read_raw_edf(psg, verbose="ERROR")
for name, n in zip(raw.ch_names, raw._raw_extras[0]["n_samps"]):
    print(f"{name:16s} {n / 30:g} Hz")

EEG Fpz-Cz       100 Hz
EEG Pz-Oz        100 Hz
EOG horizontal   100 Hz
Resp oro-nasal   1 Hz
EMG submental    1 Hz
Temp rectal      1 Hz
Event marker     1 Hz


In [3]:
PRE = {"preprocess": "bandpass", "eeg_band": (0.3, 35.0), "filter_channels": ("eeg", "eeg2")}
FIT = {"imbalance": "none"}
STAGES = ["W", "N1", "N2", "N3", "REM"]

def run(groups=()):
    cfg = {**PRE, "feature_groups": tuple(groups)}
    X, y, g = track.build_dataset(recs, cfg=cfg)
    rep = track.evaluate(X, y, g, cfg={**cfg, **FIT})
    per_subj = {r["group"]: r["cohens_kappa"] for r in rep["spread_rows"]}
    rec_ = recall_score(rep["y_true"], rep["y_pred"], labels=STAGES, average=None)
    return {"n_feat": X.shape[1], "summary": rep["summary"], "macro_f1": rep["macro_f1"],
            "accuracy": rep["accuracy"], "mean_kappa": float(np.mean(list(per_subj.values()))),
            "per_subject": per_subj, "recall": dict(zip(STAGES, rec_)), "rep": rep}

def show(name, r):
    print(f"{name}  ({r['n_feat']} features)")
    print("  ", r["summary"], f"| macro-F1 {r['macro_f1']:.3f} | accuracy {r['accuracy']:.3f}")
    print("   per subject:", {k: round(v, 3) for k, v in r["per_subject"].items()})
    print("   recall     :", {k: round(v, 2) for k, v in r["recall"].items()})

results = {"2.4 reference": run()}
show("2.4 reference", results["2.4 reference"])

2.4 reference  (11 features)
   mean cohens_kappa 0.608 (sd 0.154, range 0.431-0.697 across 3 subjects) | macro-F1 0.666 | accuracy 0.729
   per subject: {'SC400': 0.431, 'SC401': 0.696, 'SC402': 0.697}
   recall     : {'W': np.float64(0.66), 'N1': np.float64(0.42), 'N2': np.float64(0.91), 'N3': np.float64(0.64), 'REM': np.float64(0.57)}


## Rows 3.1–3.6 — one feature group at a time

In [4]:
PLAN = [("3.1 relpow", ["relpow"]), ("3.2 spindle", ["spindle"]), ("3.3 eogemg", ["eogemg"]),
        ("3.4 pz", ["pz"]), ("3.5 night_norm", ["night_norm"]), ("3.6 context", ["context"])]
for name, grp in PLAN:
    results[name] = run(grp)
    show(name, results[name])

3.1 relpow  (20 features)
   mean cohens_kappa 0.614 (sd 0.165, range 0.424-0.710 across 3 subjects) | macro-F1 0.673 | accuracy 0.733
   per subject: {'SC400': 0.424, 'SC401': 0.709, 'SC402': 0.71}
   recall     : {'W': np.float64(0.62), 'N1': np.float64(0.44), 'N2': np.float64(0.91), 'N3': np.float64(0.65), 'REM': np.float64(0.61)}
3.2 spindle  (14 features)
   mean cohens_kappa 0.674 (sd 0.107, range 0.550-0.739 across 3 subjects) | macro-F1 0.709 | accuracy 0.772
   per subject: {'SC400': 0.55, 'SC401': 0.731, 'SC402': 0.739}
   recall     : {'W': np.float64(0.71), 'N1': np.float64(0.45), 'N2': np.float64(0.93), 'N3': np.float64(0.65), 'REM': np.float64(0.7)}
3.3 eogemg  (17 features)
   mean cohens_kappa 0.645 (sd 0.159, range 0.462-0.750 across 3 subjects) | macro-F1 0.703 | accuracy 0.757
   per subject: {'SC400': 0.462, 'SC401': 0.75, 'SC402': 0.724}
   recall     : {'W': np.float64(0.71), 'N1': np.float64(0.39), 'N2': np.float64(0.92), 'N3': np.float64(0.7), 'REM': np.float64(

## Row 3.7 — combination, chosen by the rule fixed before running

A group enters the combination if it (a) raises mean κ over 2.4 **and** (b) does not lower the
worst-subject κ.

In [5]:
ref = results["2.4 reference"]
worst = min(ref["per_subject"], key=ref["per_subject"].get)
keep = []
for name, grp in PLAN:
    r = results[name]
    d_mean = r["mean_kappa"] - ref["mean_kappa"]
    d_worst = r["per_subject"][worst] - ref["per_subject"][worst]
    ok = d_mean > 0 and d_worst >= 0
    print(f"{name:16s} Δmean κ {d_mean:+.3f} | Δworst ({worst}) {d_worst:+.3f} -> {'keep' if ok else 'drop'}")
    if ok:
        keep += grp
print("combination:", keep)
results["3.7 combined"] = run(keep)
show("3.7 combined", results["3.7 combined"])

3.1 relpow       Δmean κ +0.006 | Δworst (SC400) -0.006 -> drop
3.2 spindle      Δmean κ +0.065 | Δworst (SC400) +0.119 -> keep
3.3 eogemg       Δmean κ +0.037 | Δworst (SC400) +0.032 -> keep
3.4 pz           Δmean κ +0.059 | Δworst (SC400) +0.080 -> keep
3.5 night_norm   Δmean κ +0.066 | Δworst (SC400) +0.174 -> keep
3.6 context      Δmean κ +0.020 | Δworst (SC400) -0.050 -> drop
combination: ['spindle', 'eogemg', 'pz', 'night_norm']
3.7 combined  (26 features)
   mean cohens_kappa 0.713 (sd 0.080, range 0.621-0.768 across 3 subjects) | macro-F1 0.745 | accuracy 0.795
   per subject: {'SC400': 0.621, 'SC401': 0.768, 'SC402': 0.751}
   recall     : {'W': np.float64(0.73), 'N1': np.float64(0.51), 'N2': np.float64(0.89), 'N3': np.float64(0.73), 'REM': np.float64(0.83)}


In [6]:
from report import confusion_table
for name in ["2.4 reference", "3.7 combined"]:
    r = results[name]["rep"]
    print(name); print(confusion_table(r["y_true"], r["y_pred"], r["labels"]))

2.4 reference
| true \ pred | W | N1 | N2 | N3 | REM | n |
|---|---|---|---|---|---|---|
| **W** | 0.66 | 0.12 | 0.13 | 0.08 | 0.02 | 943 |
| **N1** | 0.16 | 0.42 | 0.30 | 0.00 | 0.12 | 596 |
| **N2** | 0.02 | 0.02 | 0.91 | 0.02 | 0.03 | 2792 |
| **N3** | 0.10 | 0.00 | 0.26 | 0.64 | 0.00 | 932 |
| **REM** | 0.01 | 0.08 | 0.33 | 0.00 | 0.57 | 1028 |
3.7 combined
| true \ pred | W | N1 | N2 | N3 | REM | n |
|---|---|---|---|---|---|---|
| **W** | 0.73 | 0.22 | 0.02 | 0.00 | 0.02 | 943 |
| **N1** | 0.14 | 0.51 | 0.23 | 0.00 | 0.12 | 596 |
| **N2** | 0.01 | 0.03 | 0.89 | 0.06 | 0.02 | 2792 |
| **N3** | 0.01 | 0.00 | 0.26 | 0.73 | 0.00 | 932 |
| **REM** | 0.02 | 0.09 | 0.06 | 0.00 | 0.83 | 1028 |
